# Compose

Docker compose is a tool for defining and running multi-container Docker applications. With Docker Compose, you can use a YAML file to configure your application's services. Then, with a single command, you can create and start all the services from your configuration. This page overview core concepts of the docker compose.

In the special config you can specify the behaviour of our application as a group of containers.

Check [official instrcutions for the docker compose file](https://docs.docker.com/reference/compose-file/).

## Up/down

To get started with Docker Compose, follow these steps:

1. **Create a `docker-compose.yml` file:** This file defines your application's services, their dependencies, and configurations.
2. **Run `docker compose up`:** This command will build, start, and run all the services defined in your `docker-compose.yml` file.
3. **To stop and remove all containers and networks created by Docker Compose, use `docker compose down`.** 

Find out more in a [specific page](compose/up_down.ipynb).

---

To run docker compose, you need to specify a special config for dockercompose. The following cell defines compose config needed two services `linux1` and `linux2`.

In [4]:
#file compose.yml
services:
  linux1:
    image: alpine
  linux2:
    image: alpine

To run a defined container you need to run `compose up`, this command will automatically find `compose.yml` in the execution folder and run it.

In [7]:
docker compose up &> /dev/null

Containers we've specified to exit right after run - there's nothing specified to do for them. But in the `docker ps -a` we can still see them with `exited` status.

In [8]:
docker ps -a

CONTAINER ID   IMAGE     COMMAND     CREATED          STATUS                     PORTS     NAMES
afa8f4dbe3ef   alpine    "/bin/sh"   34 seconds ago   Exited (0) 3 seconds ago             tmp_wm0uddb-linux2-1
7f47d5834654   alpine    "/bin/sh"   34 seconds ago   Exited (0) 3 seconds ago             tmp_wm0uddb-linux1-1


To remove everything you've created, use the `docker compose down` command.

You need to run it in the folder where compose config is located - it will automatically detect what was created by this compose and delete it.

In [9]:
docker compose down

[+] down 2/3
 ✔ Container tmp_wm0uddb-linux1-1 Removed                                   0.0s
 ✔ Container tmp_wm0uddb-linux2-1 Removed                                   0.0s
 ⠋ Network tmp_wm0uddb_default    Removing                                  0.1s
[+] down 3/3
 ✔ Container tmp_wm0uddb-linux1-1 Removed                                   0.0s
 ✔ Container tmp_wm0uddb-linux2-1 Removed                                   0.0s
 ✔ Network tmp_wm0uddb_default    Removed                                   0.1s



So, after all, containres should disappear.

In [10]:
docker ps -a
rm compose.yml

CONTAINER ID   IMAGE     COMMAND   CREATED   STATUS    PORTS     NAMES


## Service configuration

The following cell shows directives that you can use to configure your service in docker compose:

| Directive         | Description |
|------------------|------------|
| `image`         | Specifies the Docker image to use for the service. |
| `build`         | Defines build configurations for creating an image. |
| `command`       | Overrides the default command for the container. |
| `entrypoint`    | Overrides the default entrypoint of the container. |
| `environment`   | Defines environment variables. |
| `env_file`      | Specifies an external file containing environment variables. |
| `ports`         | Maps container ports to the host. |
| `volumes`       | Mounts host directories or named volumes into the container. |
| `networks`      | Connects the service to specific networks. |
| `depends_on`    | Specifies dependencies on other services. |
| `restart`       | Defines the restart policy for the container. |
| `deploy`        | Specifies deployment settings (mainly for Swarm). |
| `healthcheck`   | Defines a health check command for the container. |
| `logging`       | Configures logging options for the service. |
| `ulimits`       | Sets resource limits for the container. |
| `extra_hosts`   | Adds custom host-to-IP mappings. |
| `dns`           | Specifies DNS servers for the container. |
| `dns_search`    | Defines DNS search domains. |
| `sysctls`       | Configures kernel parameters. |
| `security_opt`  | Sets security options for the container. |
| `cap_add`       | Grants additional Linux capabilities. |
| `cap_drop`      | Removes Linux capabilities. |
| `devices`       | Allows access to host devices inside the container. |
| `secrets`       | Defines secrets to be used in the container. |
| `configs`       | Specifies configuration files from Docker Swarm. |
| `tmpfs`         | Mounts a temporary filesystem inside the container. |
| `shm_size`      | Sets the size of the `/dev/shm` shared memory. |
| `privileged`    | Grants extended privileges to the container. |
| `read_only`     | Runs the container filesystem in read-only mode. |
| `stop_signal`   | Defines the system signal used to stop the container. |
| `stop_grace_period` | Specifies a timeout before forcibly stopping a container. |
| `init`          | Uses an init process inside the container. |
| `container_name` | Allows to specify a name for the result container. |
| `tty`           | Enables TTY for the container                       |
| `stdin_open`    | Enables to interuct with stdin of the container (similar to `-i` option of the `docker run`) |

For more details check:
- [Corresponding page](compose/service_configuration.ipynb) on this website.
- [Corresponding page](https://docs.docker.com/reference/compose-file/services/) of the offical documentation.

## Parametrization

You can use syntax such as `${<VARIABLE_NAME>}` to parametrize your docker compose file. The `docker compose` command inserts the values of the environment variables with the corresponding names.

---

The following cell builds a docker compose file that uses the value of an environment variable as the printed result of the container.

In [15]:
# init
# file compose.yml
services:
  test:
    image: alpine:latest
    command: echo ${SOME_VALUE}

The following cell adds the variable to the environment and runs the compose project.

In [16]:
export SOME_VALUE="hello world"
docker compose up -d &> /dev/null && docker compose logs -f

test-1 exited with code 0
test-1  | hello world


### Dot env

If some variables are not defined in the environment, the `docker compose` will look for them in the `.env` file.

---

The following cell defines a docker compose file that uses `SOME_VALUE` variable.

In [23]:
# init
# file compose.yml
services:
  test:
    image: alpine:latest
    command: echo ${SOME_VALUE}

The next code explicitly removes the `SOME_VALUE` from the environment, but keeps it in the `.env` file.

In [26]:
unset SOME_VALUE
echo "SOME_VALUE=\"value from .env\"" > .env
docker compose up -d &> /dev/null && docker compose logs -f

test-1 exited with code 0
test-1  | value from .env


The docker compose uses the corresponding variable value.